In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

E0000 00:00:1750170417.750491    1614 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750170417.756207    1614 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750170417.772552    1614 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750170417.772580    1614 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750170417.772582    1614 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750170417.772584    1614 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1750170436.292174    1614 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


# Chapter 11: Deep Learning for Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

## 11.2 Preparing Text Data

**Vectorizing** text is the process of transforming text into numeric tensors.

- First, you standardize the text to make it easier to process, such as by converting 
  it to lowercase or removing punctuation.
- You split the text into units (called tokens), such as characters, words, or groups
  of words. This is called tokenization.
- You convert each such token into a numerical vector. This will usually involve
  first indexing all tokens present in the data.

### 11.2.1 Text standardization

### 11.2.2 Text splitting (tokenization)

#### Understanding N-grams and bag-of-words

Note that the book shows example 2-gram (bigram) and 3-gram (trigram) tokenization, where for example for the 2-gram, grams of minimum length 1 or maximum
length 2 are returned.  To do this using the Tensorflow `strings.ngrams()` method requires us to generate 1-gram and 2-grams and combine:

In [4]:
text = "the cat sat on the mat".split()
tf.strings.ngrams(text, 1).numpy().tolist() + tf.strings.ngrams(text, 2).numpy().tolist()

[b'the',
 b'cat',
 b'sat',
 b'on',
 b'the',
 b'mat',
 b'the cat',
 b'cat sat',
 b'sat on',
 b'on the',
 b'the mat']

In my experience, usually when practitioners are using N-grams, they mean all sets of exactly N items.  The TensorFlow
`strings.ngrams` can create N-gram tokenization, but it defaults, as I would expect, to giving N-grams of exactly
the `ngram_width` asked for.

In [5]:
 tf.strings.ngrams(text, 2).numpy().tolist()

[b'the cat', b'cat sat', b'sat on', b'on the', b'the mat']

### 11.2.3 Vocabulary indexing

Once your text is split into tokens, you need to encode each token into a numerical
representation.

In practice, the way you’d go about it is to build an index of all terms found in the training data (the “vocabulary”), and assign a
unique integer to each entry in the vocabulary.

For example, if you wanted to do this by hand, it might look like the following, where we have
also given examples of a function that that standardizes the text `standardize()` (make lowercase and removes punctuation)
and tokenizes the text into tokens `tokenize()` (simply splits string on whitespace).


In [8]:
# each item in the list is a string which represents 1 document or sample for our corpus,
# in a real corpus of documentations, there could be a separate file for each sample document, or we might
# do some processing, like break a document into paragraphs and each paragraph is a sample, etc.
dataset = [
    "I write, erase, rewrite!",
    "Erase again, and then.",
    "A poppy blooms.",
]

In [7]:
# simplest and most widespread standardization scheme, covert to lowercase and remove punctuation
def standardize(sample):
    """Given a sample (a python string), standardize the sample.  This method performs
    basic standardization, making all characters lowercase, and removing all basic
    punctuation from the sample.

    Parameters
    ----------
    sample : str
      A sample document, should be a standard python string or string like immutable sequence.

    Returns
    -------
    standardized_sample : str
      Returns the sample document after performing standardization steps on it.
    """
    # first standardization, making lowercase is simple for python strings
    standardized_sample = sample.lower()

    # remove standard western english punctuation, would need more extensive list for real corpus
    remove_list = ".!?,;\"'" # etc.
    remove_table = str.maketrans('', '', remove_list)
    standardized_sample = standardized_sample.translate(remove_table)

    return standardized_sample

In [9]:
# tokenization can likewise be much more complex for a real corpus, here we
# just split by whitespace to tokenize a standardized sample
def tokenize(sample):
    """Tokenize a standardized sample.  This example simply tokenizes the
    sample by whitespace.

    Parameters
    ----------
    sample : str
      A sample document, should be a standard python string or string like immutable sequence.

    Returns
    -------
    tokenized_list : list
      Returns the tokenized sample as a standard python list.
    """
    tokenized_list = sample.split()
    return tokenized_list

In [10]:
# result of indexing is a dictionary, where key is the
# is the word string/token found in the corpus and the value is the
# unique integer index assigned to that word
vocabulary = {}

# it is usually to have 2 special characters in the vocabulary (at minimum)
# mask token: ignore me I'm not a word, if need to for example pad a sample to standard size
vocabulary["[MASK]"] = 0
# OOV out of vocabulary token, if need to encode a word not in original corpus vocabulary
vocabulary["[UNK]"] = 1 

# we iterate through the texts in the corpus dataset 1 by 1
for text in dataset:
    # do whatever standardization and tokenization is required for our corpus samples
    text = standardize(text)
    tokens = tokenize(text)

    # each token now represents a word in our corpus vocabulary.  If the token
    # has not been seen yet, assign it the next integer index in our vocabulary
    for token in tokens:
        if token not in vocabulary:
            vocabulary[token] = len(vocabulary)

# the resulting vocabulary dictionary
vocabulary

{'[MASK]': 0,
 '[UNK]': 1,
 'i': 2,
 'write': 3,
 'erase': 4,
 'rewrite': 5,
 'again': 6,
 'and': 7,
 'then': 8,
 'a': 9,
 'poppy': 10,
 'blooms': 11}

You can then convert that vocabulary integer into a vector encoding that can be processed like a neural network.  For example if you needed to
encode a single word with a one-hot encoding, you could do the following


In [11]:
def one_hot_encode(token):
    """Just an example of vectorizing a single token using one-hot-encoding.

    Parameters
    ----------
    token : str
        A word/token from corpus, we assume the token is in the vocabulary dictionary we
        are using.

    Returns
    -------
    vector : ndarray shape (vocabulary_size,)
        The one-hot-encoded vector representation of a text with only this word token in it.
    """
    vector = np.zeros((len(vocabulary),))
    # may result in an error if the token is not in the vocabulary, what to do then?
    token_index = vocabulary.get(token, 1) # default to OOV token if not in vocabulary
    vector[token_index] = 1
    return vector

token = 'again'
v = one_hot_encode(token)
v

array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])

### 11.2.4 Using the `TextVectorization` layer

Every step we've looked at can be implemented in Python.  Simply if you don't need much complexity in your
standardization and tokenization.  More complex for more complex corpuses.

Putting the previous example in a class, like the text does:

In [12]:
class Vectorizer:

    def standardize(self, sample):
        """Given a sample (a python string), standardize the sample.  This method performs
        basic standardization, making all characters lowercase, and removing all basic
        punctuation from the sample.
    
        Parameters
        ----------
        sample : str
          A sample document, should be a standard python string or string like immutable sequence.
    
        Returns
        -------
        standardized_sample : str
          Returns the sample document after performing standardization steps on it.
        """
        # first standardization, making lowercase is simple for python strings
        standardized_sample = sample.lower()
    
        # remove standard western english punctuation, would need more extensive list for real corpus
        remove_list = ".!?,;\"'" # etc.
        remove_table = str.maketrans('', '', remove_list)
        standardized_sample = standardized_sample.translate(remove_table)
    
        return standardized_sample

    def tokenize(self, sample):
        """Tokenize a standardized sample.  This example simply tokenizes the
        sample by whitespace.
    
        Parameters
        ----------
        sample : str
          A sample document, should be a standard python string or string like immutable sequence.
    
        Returns
        -------
        tokenized_list : list
          Returns the tokenized sample as a standard python list.
        """
        tokenized_list = sample.split()
        return tokenized_list

    def make_vocabulary(self, dataset):
        """Given a dataset corpus, create the vocabulary and inverse vocabulary
        dictionaries needed to vectorize samples for this corpus.

        Parameters
        ----------
        dataset : list
            A regular python list of the sample texts of our dataset corpus.  Samples
            are expected to be strings in this example.

        Returns
        -------
        vocabulary : dict
        inverse_vocabulary : dict
            The vocabulary and inverse vocabulary are returned, actually in this version they are
            set as member variables for this Vectorizor class.
        """
        # result of indexing is a dictionary, where key is the
        # is the word string/token found in the corpus and the value is the
        # unique integer index assigned to that word
        self.vocabulary = {}
        
        # it is usually to have 2 special characters in the vocabulary (at minimum)
        # mask token: ignore me I'm not a word, if need to for example pad a sample to standard size
        self.vocabulary["[MASK]"] = 0
        # OOV out of vocabulary token, if need to encode a word not in original corpus vocabulary
        self.vocabulary["[UNK]"] = 1
        
        # we iterate through the texts in the corpus dataset 1 by 1
        for text in dataset:
            # do whatever standardization and tokenization is required for our corpus samples
            text = self.standardize(text)
            tokens = self.tokenize(text)
        
            # each token now represents a word in our corpus vocabulary.  If the token
            # has not been seen yet, assign it the next integer index in our vocabulary
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)

        # make the inverse vocabulary dict also available, like we used in IMDB example awhile ago
        self.inverse_vocabulary = dict(
            (v, k) for k, v in self.vocabulary.items()
        )

    def encode(self, text):
        """Given a sample text, retun a list of the encoded vocabulary indexes for the whole sample
        text.

        Parameters
        -----------
        text : str
            The sample text to encode using the Vectorizer vocabulary.

        Returns
        -------
        encoding : python list like 
            Returns a list of the tokens in the text, vectorized by the vocabulary index number.
        """
        text = self.standardize(text)
        tokens = self.tokenize(text)
        encoding = [self.vocabulary.get(token, 1) for token in tokens]
        return encoding

    def decode(self, int_sequence):
        """Does the reverse of the encoding.  Giving a list of encoded vocabulary index items, return
        a string of the original encoded text.

        Parameters
        ----------
        int_sequence : a python list of int indexes
            The encoded list of vocabulary indexes for a sample vectorized using the corpus vocabulary.
            
        Returns
        -------
        decoded : str
            Returns the decoded sample string.
        """
        return " ".join(
            self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence
        )

dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]

In [13]:
from pprint import pprint

vectorizer = Vectorizer()
vectorizer.make_vocabulary(dataset)
pprint(vectorizer.vocabulary)
pprint(vectorizer.inverse_vocabulary)

{'[MASK]': 0,
 '[UNK]': 1,
 'a': 9,
 'again': 6,
 'and': 7,
 'blooms': 11,
 'erase': 4,
 'i': 2,
 'poppy': 10,
 'rewrite': 5,
 'then': 8,
 'write': 3}
{0: '[MASK]',
 1: '[UNK]',
 2: 'i',
 3: 'write',
 4: 'erase',
 5: 'rewrite',
 6: 'again',
 7: 'and',
 8: 'then',
 9: 'a',
 10: 'poppy',
 11: 'blooms'}


In [14]:
# test it
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)

decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

[2, 3, 5, 7, 1, 5, 6]
i write rewrite and [UNK] rewrite again


However you will need to do a lot more work to handle more realistic corpuses and to get good performance to encode and decode.
In practice it is better to use existing libraries like the Keras `TextVectorization` layer for this, which is fast
and efficient and covers most basic use cases well.  The `TextVectorization` can be dropped directly into a
`tf.data` pipeline like we have done previously with `tf.data.Dataset` for training examples.

Our simple example would be done using `TextVectorization` like this:

In [15]:
from tensorflow.keras.layers import TextVectorization

# configures the layer to return sequences of words encoded as integer indices.
# there are several other output modes available, which we will see in a bit.
text_vectorization = TextVectorization(
    output_mode = "int",
)

By default, the `TextVectorization` layer will use the setting "convert to lowrecase and remove punctuation" for
text standardization, and "split on whitespace" for tokenization.  

But you can provide custom function for standardization and tokenization if needed. 
**Note**: these function get handed `tf.string` tensors instead of Python strings.

For instance, if you wanted to reimplement the default standardization and tokenization using
custom functions, it would look like:

In [16]:
import re # we use re.escape() function from here in the example
import string # there is a string.punctuation member constant, who knew right?!

def custom_standardization_fn(string_tensor):
    # converts string to lowercase using equivalent tf.strings member method
    lowercase_string = tf.strings.lower(string_tensor)

    # use regular expressions, using the re regular expression library escape() function, and the
    # defination of all punctuation symbols from the standard string library
    return tf.strings.regex_replace(lowercase_string, f"[{re.escape(string.punctuation)}]", "")

# the TextVectorization class refers to this as the split function instead of more standard
# tokenization function, so we'll follow suit on their names
def custom_split_fn(string_tensor):
    return tf.strings.split(string_tensor)

text_vectorization = TextVectorization(
    output_mode="int",
    standardize=custom_standardization_fn,
    split=custom_split_fn,
)

To index the vocabulary of a text corpus, just call the `adapt()` method of the layer with a Dataset object that
yields strings.

In [18]:
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]

text_vectorization.adapt(dataset)

# you get a regular list here, ordered by the index number used in vectorization
vocabulary = text_vectorization.get_vocabulary()
pprint(vocabulary)

test_sentence = "I write, rewrite, and still rewrite again."
encoded_sentence = text_vectorization(test_sentence)
print(encoded_sentence)

# since the vocabulary is ordered by index number, this creates the inverse
# dictionary like we used before
inverse_vocab = dict(enumerate(vocabulary))
decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence)
print(decoded_sentence)

['',
 '[UNK]',
 'erase',
 'write',
 'then',
 'rewrite',
 'poppy',
 'i',
 'blooms',
 'and',
 'again',
 'a']
tf.Tensor([ 7  3  5  9  1  5 10], shape=(7,), dtype=int64)
i write rewrite and [UNK] rewrite again


**Note**: Using the `TextVectorization` layer in a `tf.data` pipeline or as part of a model.

The text discusses 2 approaches to use the `TextVectorization` layer.

1. Put the vectorization in a `tf.data` pipeline.
2. Just add it as usually the first layer, since it is a Keras layer after all, in your model.

The second option causes vectorizaiton to happen synchronously with the model being fed batches.  This
vectorization can't be parallelized on GPU, this means that it will only run on the CPU and so will
always have to wait for this to finish before GPU processing can happen.

The first option enables asynchronous processing, the vectorization can happen on the CPU, controlled
by the `tf.data` asynchronous features, while simulataneously GPU training of the model can occur.

If you only have CPU, either way should give about the same performance.  If using GPU, the second approach
will usually give much better performance.

## Summary

<font color='blue'>
    
- **Vectorizing** text is the process of transforming text into numeric tensors.
  - **standardize** text e.g. make lowercase and remove punctuation
  - **tokenize** text, split into units (usually word tokens)
  - **vectorize** create a vocabulary index that maps unique tokens to an index and convert token sequences to this index to form vectors.